# 第11章 ミニ課題 ―― β・A・Ω を変えて、予想してから確かめる

**この章でできるようになること**
- 研究の計算の条件（β・A・Ω₂）を自分で変えて、IV マップとステップ幅を計算し直せる
- 計算する前に「どう変わるか」を予想し、図と数で確かめる習慣をつける
- 点を減らした計算の結果を、研究の結論と取り違えないように書き分けられる

**例え：** この章は「**料理の味見**」です。レシピ（研究コード）の塩の量（β）や火加減（A）、材料（Ω₂）を1つだけ変えて、作る前に「たぶん辛くなる」と予想してから、少しだけ作って味見します。ただし**味見用の小皿の量**（点を減らした計算）なので、本番の鍋（研究の計算）とまったく同じ味とは限りません。

| 用語 | 意味 | 例え |
|---|---|---|
| 条件を振る | パラメータを1つずつ変えて、結果がどう変わるかを見ること | 塩の量だけ変えて味見する |
| 粗い条件 | 時間を節約するため、計算する点を減らした条件 | 味見用の小皿 |
| 標準条件 | 研究で結論を出すときに使っている条件（dt = 1/32、N_run = N_avg = 20106 など） | 本番の鍋 |
| 予想 | 計算する前に、理由をつけて結果を言っておくこと | 「たぶん辛くなる」と先に言う |
| ゼロ点 | 関数が 0 になる点。ベッセル関数 J₁(x) は x ≈ 3.83, 7.02, … で 0 | 波紋が水面と同じ高さになる場所 |

> 研究コードとのつながり：研究の `py/sim_beta.py`（β を変える）や `py/detune.py`（Ω₂ をずらす）は、この章と同じことを標準条件で、もっと広い範囲で計算しています。

In [ ]:
# 準備：教材の部品（kyozai/）を import できるようにする。どの章でも最初に1回だけ実行する。
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "kyozai").exists():   # kyozai/ があるフォルダまで上にたどる
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("教材の場所:", ROOT.name)

## 11.0 この章の約束

この章の計算は、ノートブックの中で数秒〜十数秒で終わるように、**RK4 の条件は研究と同じまま、計算する点を減らして**います。

| | 研究の標準条件 | この章 |
|---|---|---|
| dt | 1/32 | 1/32（同じ） |
| N_run = N_avg | 20106（≈ 共通周期 20π の10回分） | 20106（同じ） |
| A の点の数 | 研究の保存データによる | **17 点**（粗い） |
| 電流 | 研究の保存データによる | **V = 0.3 付近だけ** |

- 点が粗く範囲も狭いので、この章の数は、**研究の結論として使わない**でください。報告書の数値を引くときは、出どころを書き、この章の数と分けて書きます。
- 違いが出たときに「点が粗いからか、本当に違うのか」は、この章だけでは決められません（未確認）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.special import jv, jn_zeros
from kyozai.rcsj_mini import iv_map
from kyozai.step import extract
from kyozai.theory import fit, single, coherent

O1 = 0.5
N_STD = 20106          # 研究と同じ N_run = N_avg


def step_widths(beta, O2=0.8, nA=17, N=N_STD):
    """IV マップを計算し、(−1, 1) ステップの幅を A ごとに返す。
    電流は V = 0.3 付近（I = 0.16β〜0.42β）、振幅は A = 0〜6.4β。
    刻み・範囲は β に比例させている（研究の保存データの決め方と同じかは未確認）。"""
    I = np.arange(0.16 * beta, 0.42 * beta, 0.00070779 * beta)
    A = np.linspace(0, 6.4 * beta, nA)
    V = iv_map(I, A, beta, O1, O2, 1/32, N, N, 1.0)
    Vt = -O1 + O2                       # (−1, 1) の電圧
    sw = np.array([extract(I, V[:, j], Vt, 0.02, 0.04) for j in range(A.size)])
    return A, sw, I, V


t0 = time.time()
A10, sw10, I10, V10 = step_widths(10.0)
print(f"β = 10: {time.time() - t0:.1f} 秒（1回目は numba のコンパイル込み）")

β = 10 は第6〜9章と同じ条件なので、A が重なる点（A = 0, 4, 8, …）では、第6章の IV マップと同じステップ幅になるはずです。確かめます。

In [ ]:
from kyozai.data import iv_beta10
d = iv_beta10(O2=0.8)
sw_std = np.array([extract(d["I"], d["V"][:, j], 0.3, 0.02, 0.04) for j in range(d["A"].size)])
print("A が重なる点での差の最大:", np.abs(sw_std[::2] - sw10).max())

## 11.0b 予想してから実行：平均を短くしたら？

**問題：** 時間を節約したくて、捨てる区間・平均する区間を 20106 から **4021**（共通周期 20π の2回分。周期の整数倍なので、第5章の「端数」の問題はほぼない）に縮めたとします。β = 10 のステップ幅のグラフは、
- (a) ほとんど変わらない
- (b) 全体に少しずれる
- (c) A によって大きくずれるところがある

のどれになるでしょうか。

In [ ]:
t0 = time.time()
A10s, sw10s, _, _ = step_widths(10.0, N=4021)
print(f"N = 4021: {time.time() - t0:.1f} 秒")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(d["A"], sw_std, "o-", ms=4, label="N = 20106 (ch.6, 33 points)")
ax.plot(A10s, sw10s, "s--", ms=5, label="N = 4021 (short)")
ax.set_xlabel("A"); ax.set_ylabel("ΔI of (-1,1)"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("β = 10: effect of shortening N_run = N_avg")
plt.show()
print("差の最大:", np.abs(sw_std[::2] - sw10s).max().round(3))

<details><summary>解説（実行してから開く）</summary>

この計算では **(c)** になりました。第1ローブ（A ≈ 10）はほぼ重なりますが、A ≳ 20 では短い計算のステップ幅がかなり小さく、でこぼこになるところがあります（差の最大は出力のとおり）。

理由は確かめていませんが、平均する区間が短いと V の値が少しずつ揺れ、傾き |dV/dI| がしきい値 0.02 を超える点が段の途中に出て、段が途中で切れてしまうためと考えられます（推測）。

ここから言えるのは、「計算を短くして速くする」と**答えそのものが変わることがある**、ということです。この章では、研究と同じ N = 20106 を使います。
</details>

## 11.1 課題1：β を変える

**予想してから実行：** β を 10 から 6 に変えると、(−1, 1) ステップ幅のローブ（山）は、A の軸の上でどこに動くでしょうか。

ヒント：報告書の表2では、コヒーレント和でフィットした k* と β の積 k*β が、β ≥ 2.5 で 0.946〜1.018 でした（研究の保存データ。この教材では計算していない）。もし k* ≈ 1/β なら、理論の変数 x = k*A/Ω は A/β に比例します。

**先に予想をメモしてから**、次のセルを実行してください。

In [ ]:
t0 = time.time()
A6, sw6, I6, V6 = step_widths(6.0)
print(f"β = 6: {time.time() - t0:.1f} 秒")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(A10, sw10, "o-", label="β = 10")
ax[0].plot(A6, sw6, "s-", label="β = 6")
ax[0].set_xlabel("A"); ax[0].set_title("plotted against A")
ax[1].plot(A10 / 10, sw10 / sw10.max(), "o-", label="β = 10")
ax[1].plot(A6 / 6, sw6 / sw6.max(), "s-", label="β = 6")
ax[1].set_xlabel("A / β"); ax[1].set_title("plotted against A/β (height normalized)")
for a_ in ax:
    a_.set_ylabel("ΔI of (-1,1)"); a_.legend(); a_.grid(alpha=.3)
fig.tight_layout(); plt.show()

for beta, A_, sw_ in [(10.0, A10, sw10), (6.0, A6, sw6)]:
    r, k, sc = fit(sw_, A_, coherent, -1, 1)
    print(f"β = {beta:4.1f}: コヒーレント和のフィット r = {r:.3f}, k* = {k:.4f}, k*β = {k * beta:.3f}")

<details><summary>解説（実行してから開く）</summary>

- 左の図（横軸 A）では、β = 6 の山は β = 10 より**小さい A のほうに寄って**います。A の範囲も 0〜6.4β にしているので、β = 6 では A = 0〜38.4 です。
- 右の図（横軸 A/β、高さをそろえたもの）では、この計算では2本がほとんど重なりました。「ローブの位置は A/β で決まる」という見方が、β = 6 と 10 のこの計算では成り立っている、ということです。
- 下の出力では、k*β は β = 10 で 1.005、β = 6 で 1.006 と、どちらも 1 に近くなりました。報告書の表2の範囲（0.946〜1.018）に入るかどうかは、この章の数だけで「確かめた」とは言えません（点が粗く、β も2つだけ）。
</details>

## 11.2 課題2：A を変える ―― 谷の位置を理論から予想する

**予想してから実行：** 単項理論 |J₋₁(x₁) J₁(x₂)|（x₁ = kA/Ω₁、x₂ = kA/Ω₂）では、J₁ が 0 になるところでステップ幅も 0 になります。k = 1/β = 0.1（β = 10）とすると、**最初の2つのゼロ点**は A がいくつのところでしょうか。J₁ のゼロ点 x ≈ 3.83 を使って、x₁ と x₂ それぞれについて計算してください。

In [ ]:
j1_zeros = jn_zeros(1, 2)           # J1 の最初の2つのゼロ点
k = 0.1
print("J1 のゼロ点:", np.round(j1_zeros, 3))
print("x1 = kA/Ω1 がゼロ点になる A:", np.round(j1_zeros * O1 / k, 1))
print("x2 = kA/Ω2 がゼロ点になる A:", np.round(j1_zeros * 0.8 / k, 1))

zeros_A = np.sort(np.concatenate([j1_zeros * O1 / k, j1_zeros * 0.8 / k]))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(d["A"], sw_std, "o-", c="#cc0000", ms=4, label="numerical ΔI (standard, ch.6)")
Af = np.linspace(0, 64, 600)
ax.plot(Af, single(-1, 1, Af, k) / single(-1, 1, Af, k).max() * sw_std.max(), "k--", lw=1, label="single, k = 0.1 (scaled)")
for zA in zeros_A:
    ax.axvline(zA, color="gray", ls=":", lw=1)
ax.set_xlabel("A"); ax.set_ylabel("ΔI of (-1,1)"); ax.legend(fontsize=9); ax.grid(alpha=.3)
ax.set_title("β = 10: zeros of the single-term theory (dotted)")
plt.show()

<details><summary>解説（実行してから開く）</summary>

- x₁ = kA/Ω₁ = 3.83 → A = 3.83 × 0.5 / 0.1 ≈ **19.2**、x₂ = kA/Ω₂ = 3.83 → A = 3.83 × 0.8 / 0.1 ≈ **30.7** です。その次は x₁ のゼロ点 7.02 から A ≈ 35.1 です。
- 図の点線（単項理論のゼロ点）と、データ（赤）の谷の位置がどれくらい合っているかを見てください。データは A = 2 刻みなので、谷の位置は 2 くらいの幅でしか読めません。
- 第9章で見たように、コヒーレント和では直角の項が単項のゼロ点を「埋める」ので、データの谷が 0 まで下がらない場所があっても不思議ではありません。どの谷がどれだけ埋まるかは、この章では調べていません（未確認）。
</details>

## 11.3 課題3：Ω₂ を変える ―― 段の電圧を予想する

**予想してから実行：** Ω₂ を 0.8 から **0.7** に変えると、(−1, 1) ステップは V がいくつのところに出るでしょうか。また、Ω₁ : Ω₂ = 0.5 : 0.7 は簡単な整数の比になるでしょうか。

下のセルは β = 10、A = 10 の IV 曲線を1本だけ計算します。

In [ ]:
O2c = 0.7
Ic = np.arange(1.0, 4.2, 0.0070779)   # V = 0.2 の段が端で切れないよう、下を広めにとる
t0 = time.time()
Vc = iv_map(Ic, np.array([10.0]), 10.0, O1, O2c, 1/32, N_STD, N_STD, 1.0)[:, 0]
print(f"{time.time() - t0:.1f} 秒")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(Ic, Vc, lw=1.2, label="Ω2 = 0.7, A = 10")
for Vs in [0.1, 0.2, 0.3, 0.4]:
    ax.axhline(Vs, color="r", ls=":", lw=0.8)
ax.set_xlabel("I"); ax.set_ylabel("V"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("β = 10, Ω1 = 0.5, Ω2 = 0.7 (this chapter)")
plt.show()

w = extract(Ic, Vc, -O1 + O2c, 0.02, 0.04)
print(f"(−1, 1) の電圧 = {-O1 + O2c:.1f}、そのステップ幅 ΔI = {w:.3f}")

<details><summary>解説（実行してから開く）</summary>

- (−1, 1) の電圧は −0.5 + 0.7 = **0.2** です。図でも V = 0.2 に長い段が出ています。V = 0.3 や 0.4 にも段があります（たとえば (2, −1) は 1 − 0.7 = 0.3）。
- 0.5 : 0.7 = **5 : 7** で、簡単な整数の比です（通約）。7Ω₁ = 5Ω₂ = 3.5 なので、(n + 7j, m − 5j) が同じ電圧に届きます。たとえば (6, −4) も V = 3 − 2.8 = 0.2 です。
- 共通周期は、2π/0.5 = 4π と 2π/0.7 = 20π/7 がそろう **20π** で、Ω₂ = 0.8 のときと同じです。そのため、この章の N_STD（≈ 20π の10回分）のままで平均できます。
- 0.5 : 0.7 のときに第3ローブがどうなるかは、この章では計算していません（未確認）。それを調べるのが下の自由課題の一例です。
</details>

## 11.4 自由課題：自分で条件を決める

下のセルの `beta`・`O2` を変えて、(−1, 1) のステップ幅を計算してください。**実行する前に**、次の3つを書いておきましょう。

1. 何を変えるか（1つだけ）
2. どうなると予想するか、その理由
3. 予想が外れたら、何を疑うか（点が粗いせい？ 電流の範囲が足りない？ 窓 ±0.04 に別の段が入っている？）

注意：
- `step_widths` は電流を I = 0.16β〜0.42β だけ計算します。Ω₂ を大きく変えて (−1, 1) の電圧が 0.3 から離れると、段が範囲の外に出ることがあります。そのときは IV 曲線を描いて確かめてください（11.3 のように）。
- 1回の計算は数秒〜十数秒です。β を大きくすると点の数は同じでも、時間はあまり変わりません。

In [ ]:
# ここを変える
beta = 8.0
O2 = 0.8

t0 = time.time()
A_, sw_, I_, V_ = step_widths(beta, O2=O2)
print(f"β = {beta}, Ω2 = {O2}: {time.time() - t0:.1f} 秒")

r_s, k_s, sc_s = fit(sw_, A_, single, -1, 1, O2=O2)
r_c, k_c, sc_c = fit(sw_, A_, coherent, -1, 1, O2=O2)
print(f"単項         : r = {r_s:.3f}, k*β = {k_s * beta:.3f}")
print(f"コヒーレント和: r = {r_c:.3f}, k*β = {k_c * beta:.3f}   （Ω1:Ω2 = 5:8 のときの式。O2 を変えたら式が合っているか注意）")

Af = np.linspace(0, A_.max(), 400)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(A_, sw_, "o", c="#cc0000", label="numerical (this chapter)")
ax.plot(Af, sc_s * single(-1, 1, Af, k_s, O2=O2), "k--", label=f"single r = {r_s:.3f}")
ax.set_xlabel("A"); ax.set_ylabel("ΔI of (-1,1)"); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f"β = {beta}, Ω2 = {O2}")
plt.show()

`kyozai/theory.py` の `coherent` は、**Ω₁ : Ω₂ = 5 : 8 のときの (n + 8j, m − 5j)** を足す式です（`P=8, Q=5`）。Ω₂ を変えたときは、同じ電圧に届く組が変わるので、`coherent(..., P=?, Q=?)` を自分で決める必要があります（たとえば 5 : 7 なら P = 7、Q = 5）。比が簡単な整数にならないとき（0.81 など）は、足す相手がありません。

## 練習問題

1. 11.1 を β = 4 でもやってみて、3本を横軸 A/β で重ねてください。
2. 11.3 の Ω₂ = 0.7 で、`step_widths(10.0, O2=0.7)` を使ってステップ幅の A 依存性を求め、`fit(..., coherent, -1, 1, O2=0.7, P=7, Q=5)` と単項でフィットして r を比べてください。

<details><summary>答え</summary>

```python
# 1.
A4, sw4, _, _ = step_widths(4.0)
fig, ax = plt.subplots()
for b_, A_b, s_b in [(10, A10, sw10), (6, A6, sw6), (4, A4, sw4)]:
    ax.plot(A_b / b_, s_b / s_b.max(), "o-", label=f"β = {b_}")
ax.set_xlabel("A / β"); ax.legend(); plt.show()
# 2.
A7, sw7, _, _ = step_widths(10.0, O2=0.7)
print(fit(sw7, A7, single, -1, 1, O2=0.7)[:2])
print(fit(sw7, A7, coherent, -1, 1, O2=0.7, P=7, Q=5)[:2])
```
どちらも、出てきた数は**この章の粗い計算の結果**として書きます。β = 4 では電流の範囲 0.16β〜0.42β に段が入っているか、まず IV 曲線で確かめてください。2. で単項とコヒーレント和の r にどれだけ差が出るかは、この教材では確かめていません（未確認）。
</details>

## まとめ
- 条件は1つずつ変え、**計算の前に予想を書く**。外れたら、何を疑うかも書く。
- β を変えたときは、横軸を A/β にして比べると、ローブの位置の変わり方が見やすい（この章の計算での見方）。
- 単項理論のゼロ点は `jn_zeros` と x = kA/Ω から予想できる。
- Ω₂ を変えると段の電圧 V = nΩ₁ + mΩ₂ が変わり、通約の比（5:8、5:7 など）で「同じ電圧に届く組」も変わる。
- 計算を短くすると答えが変わることがある。点を減らした結果は、研究の結論と分けて書く。

## 全11章を終えて

ここまでで、研究コード `shapiro-research/py/` に出てくる道具はひととおり使い、自分で条件を変えて回すところまで来ました。次は研究リポジトリの `py/sim_beta.py` や `py/detune.py` を、この教材の第6・11章と見比べながら読んでみてください。